# PlasticWise AI - 01_preprocessing
**Week 2:** data cleaning, EDA graphs (bar, pie, box, scatter) aur stratified 70/15/15 split.

Pehle Runtime > T4 GPU (zaroori nahi is notebook ke liye, lekin theek hai). Cells upar se neeche ek ek karke run karo.
**Note:** Kaggle ka folder read-only hai, isliye kharab/duplicate files *move* nahi hongi. Unki list `rejected_files.csv` mein save hogi aur clean list se bahar rahengi (original files safe).

## Cell 1: Setup (Drive + dataset dobara)
Colab session naya ho to ye zaroori hai.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
from pathlib import Path

BASE = '/content/drive/MyDrive/PlasticWise'
for sub in ['data', 'results', 'results/eda', 'models', 'app']:
    os.makedirs(f'{BASE}/{sub}', exist_ok=True)

import kagglehub
path = kagglehub.dataset_download("sumn2u/garbage-classification-v2")

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def find_data_dir(root):
    best, best_n = None, 0
    for d in [Path(root)] + [p for p in Path(root).rglob('*') if p.is_dir()]:
        classes = [s for s in d.iterdir() if s.is_dir() and any(f.suffix.lower() in IMG_EXT for f in s.iterdir() if f.is_file())]
        if len(classes) > best_n:
            best, best_n = d, len(classes)
    return best

DATA_DIR = find_data_dir(path)
print("DATA_DIR =", DATA_DIR)

## Cell 2: Har image ko check karo (khulti hai? size? brightness? duplicate hash?)
Taqreeban 1-3 minute lagte hain.

In [ ]:
import hashlib
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm

def md5_of(fp, chunk=1 << 20):
    h = hashlib.md5()
    with open(fp, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

class_dirs = sorted(p for p in DATA_DIR.iterdir() if p.is_dir())
files = sorted(f for c in class_dirs for f in c.rglob('*') if f.is_file() and f.suffix.lower() in IMG_EXT)
print("Total files found:", len(files))

records = []
for fp in tqdm(files):
    rel = fp.relative_to(DATA_DIR)
    rec = {'filepath': str(fp), 'relpath': str(rel), 'label': rel.parts[0],
           'size_kb': round(fp.stat().st_size / 1024, 1), 'status': 'ok',
           'mode': None, 'width': None, 'height': None, 'brightness': None,
           'md5': None, 'error': ''}
    try:
        with Image.open(fp) as im:
            im.verify()                      # file theek hai ya nahi
        with Image.open(fp) as im:
            rec['mode'] = im.mode
            rec['width'], rec['height'] = im.size
            g = im.convert('L').resize((64, 64))
            rec['brightness'] = float(np.asarray(g).mean())
        rec['md5'] = md5_of(fp)
    except Exception as e:
        rec['status'] = 'corrupt'
        rec['error'] = str(e)[:80]
    records.append(rec)

df = pd.DataFrame(records)
print(df['status'].value_counts())
print("\nImage modes:\n", df['mode'].value_counts(dropna=False))

## Cell 3: Duplicates aur ghalat label dhoondo

In [ ]:
# 1) Wohi image alag-alag classes mein ho to label conflict hai (dono taraf se hata dete hain)
ok = df['status'] == 'ok'
n_labels = df[ok].groupby('md5')['label'].nunique()
conflict_md5 = n_labels[n_labels > 1].index
df.loc[ok & df['md5'].isin(conflict_md5), 'status'] = 'label_conflict'

# 2) Same class mein exact duplicates: pehli copy rakho, baqi reject
ok = df['status'] == 'ok'
dup_mask = ok & df.duplicated('md5', keep='first')
df.loc[dup_mask, 'status'] = 'duplicate'

print(df['status'].value_counts())
print("Cross-class (label conflict) hashes:", len(conflict_md5))

rejected = df[df['status'] != 'ok']
rejected.to_csv(f'{BASE}/data/rejected_files.csv', index=False)
clean = df[df['status'] == 'ok'].reset_index(drop=True)
df.to_csv(f'{BASE}/data/image_stats_all.csv', index=False)
clean.to_csv(f'{BASE}/data/image_stats_clean.csv', index=False)
print("Saved rejected_files.csv, image_stats_all.csv, image_stats_clean.csv")

## Cell 4: Before / after table
Is table ko report mein daalna.

In [ ]:
before = df.groupby('label').size().rename('before')
after = clean.groupby('label').size().rename('after')
tbl = pd.concat([before, after], axis=1).fillna(0).astype(int)
tbl['removed'] = tbl['before'] - tbl['after']
tbl['percent_after'] = (tbl['after'] / tbl['after'].sum() * 100).round(2)
print(tbl)
print("\nTOTAL before:", tbl['before'].sum(), "| after:", tbl['after'].sum(), "| removed:", tbl['removed'].sum())
tbl.to_csv(f'{BASE}/results/class_counts_before_after.csv')

## Cell 5: EDA graphs (6 figures)
Har graph `results/eda/` mein save hota hai.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
EDA = f'{BASE}/results/eda'
order = sorted(clean['label'].unique())

# 1) Bar plot: images per class
counts = clean['label'].value_counts().reindex(order)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(order, counts.values, color=sns.color_palette('tab10', len(order)))
ax.bar_label(bars)
ax.set_title('Images per class (after cleaning)'); ax.set_ylabel('Count')
plt.xticks(rotation=30); plt.tight_layout()
plt.savefig(f'{EDA}/01_bar_class_counts.png', dpi=200); plt.show()

# 2) Pie chart: share of each class
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(counts.values, labels=order, autopct='%1.1f%%', startangle=90,
       colors=sns.color_palette('tab10', len(order)))
ax.set_title('Class share (%)'); plt.tight_layout()
plt.savefig(f'{EDA}/02_pie_class_share.png', dpi=200); plt.show()

# 3) Box plot: width and height per class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=clean, x='label', y='width', order=order, ax=axes[0]); axes[0].set_title('Image width per class')
sns.boxplot(data=clean, x='label', y='height', order=order, ax=axes[1]); axes[1].set_title('Image height per class')
for a in axes: a.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.savefig(f'{EDA}/03_box_width_height.png', dpi=200); plt.show()

# 4) Scatter: width vs height (3000 sample). Chhota jitter taake overlap wale points dikhein.
samp = clean.sample(min(3000, len(clean)), random_state=42)
rng = np.random.default_rng(42)
fig, ax = plt.subplots(figsize=(8, 7))
sns.scatterplot(x=samp['width'] + rng.normal(0, 1.5, len(samp)),
                y=samp['height'] + rng.normal(0, 1.5, len(samp)),
                hue=samp['label'], hue_order=order, alpha=0.6, s=18, ax=ax)
ax.set_xlabel('width (px, small jitter added)'); ax.set_ylabel('height (px, small jitter added)')
ax.set_title('Width vs height by class'); plt.tight_layout()
plt.savefig(f'{EDA}/04_scatter_width_height.png', dpi=200); plt.show()

# 4b) Scatter: brightness vs file size (agar sab images ek jaisi size ki hon to ye zyada kaam ka hai)
fig, ax = plt.subplots(figsize=(8, 7))
sns.scatterplot(data=samp, x='brightness', y='size_kb', hue='label', hue_order=order, alpha=0.6, s=18, ax=ax)
ax.set_title('Brightness vs file size by class'); plt.tight_layout()
plt.savefig(f'{EDA}/04b_scatter_brightness_filesize.png', dpi=200); plt.show()

# 5) Sample grid: 5 random images per class
fig, axes = plt.subplots(len(order), 5, figsize=(10, 2 * len(order)))
for r, cls in enumerate(order):
    pick = clean[clean['label'] == cls].sample(5, random_state=1)
    for c, fp in enumerate(pick['filepath']):
        axes[r, c].imshow(Image.open(fp).convert('RGB')); axes[r, c].axis('off')
        if c == 0: axes[r, c].set_title(cls, loc='left', fontsize=10)
plt.tight_layout(); plt.savefig(f'{EDA}/05_sample_grid.png', dpi=150); plt.show()

# 6) Box plot: mean brightness per class
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=clean, x='label', y='brightness', order=order, ax=ax)
ax.set_title('Mean brightness per class'); plt.xticks(rotation=30); plt.tight_layout()
plt.savefig(f'{EDA}/06_box_brightness.png', dpi=200); plt.show()

## Cell 6: Insight sentences ke liye asal numbers
Ye numbers apne graphs se milao, phir 6-8 sentences khud likho.

In [ ]:
mx, mn = counts.idxmax(), counts.idxmin()
print(f"Sab se bari class : {mx} ({counts[mx]})")
print(f"Sab se chhoti class: {mn} ({counts[mn]})")
print(f"Imbalance ratio    : {counts[mx] / counts[mn]:.1f} : 1")
print(f"Plastic            : {counts['plastic']} images ({counts['plastic'] / counts.sum() * 100:.1f}%)")
print("\nUnique image sizes :", clean.groupby(['width', 'height']).ngroups)
print(clean.groupby(['width', 'height']).size().sort_values(ascending=False).head(5))
print("\nBrightness (mean) per class:")
print(clean.groupby('label')['brightness'].mean().round(1).sort_values())

## Cell 7: Stratified 70/15/15 split + class weights

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

data = clean[['filepath', 'label', 'relpath']].reset_index(drop=True)
train, temp = train_test_split(data, test_size=0.30, stratify=data['label'], random_state=42)
val, test = train_test_split(temp, test_size=0.50, stratify=temp['label'], random_state=42)

# Koi file do splits mein nahi honi chahiye
assert not (set(train.filepath) & set(val.filepath))
assert not (set(train.filepath) & set(test.filepath))
assert not (set(val.filepath) & set(test.filepath))
print("No overlap between splits: OK")

for name, d in [('train', train), ('val', val), ('test', test)]:
    d.to_csv(f'{BASE}/data/{name}.csv', index=False)
    print(f"{name:5s}: {len(d)} images")

print("\nClass share per split (%):")
print(pd.DataFrame({n: d['label'].value_counts(normalize=True).reindex(order) * 100
                    for n, d in [('train', train), ('val', val), ('test', test)]}).round(1))

# Class weights (sirf train se)
class_names = order
w = compute_class_weight('balanced', classes=np.array(class_names), y=train['label'])
class_weights = {i: float(x) for i, x in enumerate(w)}
print("\nClass weights:")
for i, c in enumerate(class_names):
    print(f"  {i} {c:10s} {class_weights[i]:.3f}")

with open(f'{BASE}/data/class_weights.json', 'w') as f: json.dump(class_weights, f, indent=2)
with open(f'{BASE}/data/class_names.json', 'w') as f: json.dump(class_names, f, indent=2)
print("\nSaved: train.csv, val.csv, test.csv, class_weights.json, class_names.json")

## Week 2 checklist
- [ ] Cell 4 ka before/after table dekh liya
- [ ] `results/eda/` mein 6 (+1) graphs bane
- [ ] Cell 6 ke numbers se 6-8 insight sentences likhe
- [ ] `train.csv`, `val.csv`, `test.csv`, `class_weights.json` Drive mein hain
- [ ] Notebook + `results/eda` GitHub par commit